In [ ]:
import numpy as np
import pandas as pd


from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score


from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

In [ ]:
X, y = make_classification(
n_samples=1500,
n_features=20,
n_informative=6,
n_redundant=6,
class_sep=1.0,
random_state=42)

X_train, X_test, y_train, y_test = train_test_split(
X, y, test_size=0.25, random_state=42, stratify=y)

Accuracy → datasets balanceados


F1 → clases desbalanceadas

In [ ]:
def evaluate(model, X_tr, y_tr, X_te, y_te):
    model.fit(X_tr, y_tr)
    return {
        "train_f1": f1_score(y_tr, model.predict(X_tr)),
        "test_f1": f1_score(y_te, model.predict(X_te))
    }

In [ ]:
# Modelo simple
baseline = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=2000)) ])


res_baseline = evaluate(baseline, X_train, y_train, X_test, y_test)
res_baseline

In [ ]:
# overfitting  arbol muy profundo → memoriza datos.
    overfit_model = DecisionTreeClassifier(
    max_depth=None,
    min_samples_leaf=1,
    random_state=42 )


res_overfit = evaluate(overfit_model, X_train, y_train, X_test, y_test)
res_overfit

overfitting implica:
- F1 muy alto en train
- Caída fuerte en test

In [ ]:
pd.DataFrame([
    {"model": "LogisticRegression", **res_baseline},
    {"model": "DecisionTree (overfit)", **res_overfit}])

In [ ]:
# Validación cruzada
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(baseline, X_train, y_train, cv=cv, scoring="f1")
cv_scores.mean(), cv_scores.std()

Media → rendimiento esperado


Desviación → estabilidad del modelo

In [ ]:
# Reducir overfitting: limitar complejidad
controlled_tree = DecisionTreeClassifier(
    max_depth=6,
    min_samples_leaf=10,
    random_state=42 )

res_controlled = evaluate(controlled_tree, X_train, y_train, X_test, y_test)
res_controlled

In [ ]:
# Reducir overfitting: ensemble (Random Forest)
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1 )

res_rf = evaluate(rf, X_train, y_train, X_test, y_test)
res_rf

In [ ]:
pd.DataFrame([
    {"model": "Baseline", **res_baseline},
    {"model": "Overfit tree", **res_overfit},
    {"model": "Controlled tree", **res_controlled},
    {"model": "Random Forest", **res_rf} ])